In [ ]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from skimage.io import imread
from tqdm import tqdm

# Paths
DATA_DIR = os.path.expanduser("~/cp-anemia-detection/data/fingernail-anemia")
CSV_PATH = os.path.join(DATA_DIR, "metadata.csv")
metadata = pd.read_csv(CSV_PATH)
YOLO_DATASET_DIR = os.path.join(os.path.expanduser("~/cp-anemia-detection/data"), "fingernail-anemia-yolo")
IMG_DIR = os.path.join(YOLO_DATASET_DIR, "images")
LBL_DIR = os.path.join(YOLO_DATASET_DIR, "labels")

metadata["HB_LEVEL_GperDeciL"] = metadata["HB_LEVEL_GperL"]/10
metadata['NAIL_BOUNDING_BOXES'] = metadata['NAIL_BOUNDING_BOXES'].apply(json.loads)
metadata['SKIN_BOUNDING_BOXES'] = metadata['SKIN_BOUNDING_BOXES'].apply(json.loads)

# Create directory structure
for split in ["train", "val"]:
    os.makedirs(os.path.join(IMG_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LBL_DIR, split), exist_ok=True)

# Split data
train_df, val_df = train_test_split(metadata, test_size=0.2, random_state=42)

def convert_and_save(df, split):
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split} set"):

        label = 'Anemic' if row.HB_LEVEL_GperDeciL < 12.0 else 'Non-anemic'
        image_path = os.path.join(DATA_DIR ,label, f'{row.PATIENT_ID}.jpg')

        if not os.path.isfile(image_path):
            print(f"Image not found: {image_path}")
            continue

        image = imread(image_path)
        height, width = image.shape[:2]

        label_path = os.path.join(LBL_DIR, split, f"{row.PATIENT_ID}.txt")

        with open(label_path, 'w') as f:
            for bbox in row["NAIL_BOUNDING_BOXES"]:
                if not isinstance(bbox, list) or len(bbox) != 4:
                    continue  # Skip malformed bbox

                top, left, bottom, right = bbox
                x_center = ((left + right) / 2) / width
                y_center = ((top + bottom) / 2) / height
                bbox_width = (right - left) / width
                bbox_height = (bottom - top) / height

                f.write(f"0 {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}\n")

        # Copy image to correct split folder
        dest_path = os.path.join(IMG_DIR, split, f"{row.PATIENT_ID}.jpg")
        shutil.copy(image_path, dest_path)

# Rerun with corrected assumptions
convert_and_save(train_df, "train")
convert_and_save(val_df, "val")

In [ ]:
data_yaml = """
path: {}
train: images/train
val: images/val
names:
  0: fingernail
""".format(YOLO_DATASET_DIR)

with open(os.path.join(YOLO_DATASET_DIR, "fingernail.yaml"), "w") as f:
    f.write(data_yaml)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Train
model.train(
    data=os.path.join(YOLO_DATASET_DIR, "fingernail.yaml"),
    epochs=100,
    imgsz=640,
    batch=16,
    project="yolo_fingernail_detection",
    name="yolov8n_fingernail",
    exist_ok=True
)

In [ ]:
model = YOLO("model_compression/yolov8n_fingernail/weights/best.pt")
results = model.predict(source=os.path.join(YOLO_DATASET_DIR, "images/val"), save=True)

In [44]:
resource_opt_config = {
    "epochs": 50,                   # Start small; tune based on training curve
    "imgsz": 416,                   # Smaller image = faster and smaller model
    "batch": 16,                    # Keep this low to prevent memory overload
    "lr0": 0.003,                   # Lower LR to prevent overshooting
    "weight_decay": 0.0002,         # Lower = less regularization
    "momentum": 0.9,
    "warmup_epochs": 3.0,
    "optimizer": "SGD",             # Lightweight vs. AdamW
    "hsv_h": 0.01,                  # Reduce augmentations — more stable for small models
    "hsv_s": 0.3,
    "hsv_v": 0.3,
    "scale": 0.3,
    "translate": 0.0,               # Disable unnecessary augmentations
    "fliplr": 0.5,                  # Horizontal flip only
    "mosaic": 0.0,                  # Turn off mosaic (not ideal for small datasets)
    "mixup": 0.0,                   # Disable mixup
    "project": "fingernail_resource_opt",
    "name": "yolov8n_optimized",
    "exist_ok": True
}

In [45]:
model.train(
    data=os.path.join(YOLO_DATASET_DIR, "fingernail.yaml"),
    **resource_opt_config
)

Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/fingernail.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=416, save=True, save_period=-1, cache=False, device=None, workers=8, project=fingernail_resource_opt, name=yolov8n_optimized, exist_ok=True, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=F

100%|██████████| 5.35M/5.35M [00:00<00:00, 12.1MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6523.0±2785.9 MB/s, size: 383.0 KB)


train: Scanning /home/sebastian-cruz6/cp-anemia-detection/data/fingernail-anemia-yolo/labels/train.cache... 200 images, 0 backgrounds, 0 corrupt: 100%|██████████| 200/200 [00:00<?, ?it/s]


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2781.6±2579.1 MB/s, size: 371.3 KB)


...

                   all         50        150      0.993          1      0.995      0.704



50 epochs completed in 0.007 hours.
Optimizer stripped from fingernail_resource_opt/yolov8n_optimized/weights/last.pt, 6.2MB
Optimizer stripped from fingernail_resource_opt/yolov8n_optimized/weights/best.pt, 6.2MB

Validating fingernail_resource_opt/yolov8n_optimized/weights/best.pt...
Ultralytics 8.3.115 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:00<00:00, 24.79it/s]


                   all         50        150      0.993          1      0.995      0.717
Speed: 0.0ms preprocess, 0.2ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to fingernail_resource_opt/yolov8n_optimized


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7859c6fa5d90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Load best model
model = YOLO("fingernail_resource_opt/yolov8n_optimized/weights/best.pt")

# Export to ONNX, TorchScript, or TFLite
# model.export(format="onnx")       # For edge AI engines
# model.export(format="torchscript") # For PyTorch-based mobile
# model.export(format="tflite")      # For Android / TF Lite runtimes